# Qwen3.6-27B PWMV: Probe-Weighted Voting on New Dense Flagship

Target: validate that PWMV wrapper transfers to **Qwen3.6-27B** (dense, released 2026-04-21).

## Expected delta

Qwen team reported **SuperGPQA 66.0** on 27B (vs 64.7 on 35B-A3B). If PWMV delivers +4-6pp here like on 35B-A3B, **PWMV on 27B → ~70-72%**, approaching Claude 4.5 Opus (70.6).

## Strategy

- Use same SuperGPQA questions from `Qwen3.6-35B-A3B-mcr-stage-b` (just the question metadata, not activations)
- Generate 80 greedy rollouts through **27B** → new (L11-residual, correct) corpus
- Train 27B-specific L11 probe (d=5120)
- Eval 20 held-out prompts: 4-method (greedy / naive / bo4 / PWMV)

## Pipeline (crash-safe: each stage saves → uploads before next)

1. Setup + auth
2. Load Qwen3.6-27B (~4 min download, dense model, 54GB bf16)
3. Stage B question metadata
4. **Forward + label 80 rollouts** (~3h, crash-saves every 10)
5. **Upload corpus to HF immediately** (crash-safety)
6. Train L11 probe + eval AUROC
7. **Upload probe immediately**
8. PWMV eval helpers
9. Run 4-method eval on 20 held-out (~3h, crash-saves every 3)
10. Upload summary + README + charts

Total budget ~6-7h. Designed to survive crashes via immediate HF uploads at each checkpoint.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5' in CONFIG_MAPPING_NAMES
except Exception: ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from safetensors import safe_open
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from huggingface_hub import snapshot_download, HfApi, create_repo
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/qwen27b_pwmv'); OUT.mkdir(exist_ok=True)
HF_OUT = 'caiovicentino1/qwen36-27b-deepconf-probe'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
MODEL_ID = 'Qwen/Qwen3.6-27B'

PROBE_LAYER = 11  # GA layer (position 3 in block-of-4)
D_MODEL = 5120    # hidden_size 27B dense
N_TRAIN_ROLLOUTS = 80
N_EVAL_PROMPTS = 20
N_TRACES = 4
TEMP = 0.7
MAX_NEW = 3500
FORCE_SUFFIX = '\n</think>\n\nFinal answer: \\boxed{'

print('setup done (27B dense target)')

## Cell 2 — Load Qwen3.6-27B dense

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Qwen3.6-27B loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

# Hook L11 output (should be GA layer: positions 3,7,11,15,... in 64-layer stack)
captured_l11 = {'chunks': []}
def l11_hook(module, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    captured_l11['chunks'].append(h[:, -1, :].detach().float().cpu())

layer_L = get_layer_module(model, PROBE_LAYER)
h_handle = layer_L.register_forward_hook(l11_hook)
print(f'OK L{PROBE_LAYER} hook on {type(layer_L).__name__}')

## Cell 3 — Load SuperGPQA question metadata from Stage B

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

print(f'{len(questions)} unique SuperGPQA questions')

# Split train/eval (seed different from 35B eval to avoid contamination)
random.seed(42)
random.shuffle(questions)
train_questions = questions[:N_TRAIN_ROLLOUTS]
eval_questions = questions[N_TRAIN_ROLLOUTS:N_TRAIN_ROLLOUTS + N_EVAL_PROMPTS]
print(f'Train {len(train_questions)} + Eval {len(eval_questions)} = {len(train_questions)+len(eval_questions)}')

## Cell 4 — Forward 80 greedy rollouts through 27B, capture L11 + label correctness

This is the LONG step (~3h, 2-3 min per rollout). Crash-safe: saves every 10 rollouts.

In [ ]:
def format_prompt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = (f'Answer the following multiple-choice question. '
               f'Reason step by step, then put the letter in \\boxed{{}}.\n\n'
               f'Question: {q}\n\nOptions:\n{choices}')
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True, enable_thinking=True)

def extract_answer(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    for pattern in [r'\\boxed\{([A-J])\}',
                    r'[Aa]nswer\s*(?:is\s*)?[:=]?\s*\*?\*?\(?([A-J])\)?\*?\*?',
                    r'^\s*\(?([A-J])\)?\s*\.?\s*$']:
        m = re.search(pattern, post, re.MULTILINE)
        if m: return m.group(1).upper()
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    tail = post[-300:] if post else text[-300:]
    letters = re.findall(r'\b([A-J])\b', tail)
    return letters[-1] if letters else None

def force_close(full_ids, prompt_len):
    gen = tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=False)
    if '</think>' in gen:
        return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True)
    full = tok.decode(full_ids.tolist(), skip_special_tokens=False) + FORCE_SUFFIX
    ctx = tok(full, return_tensors='pt', add_special_tokens=False).input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ctx, max_new_tokens=15, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    suf = tok.decode(out[0, ctx.shape[1]:].tolist(), skip_special_tokens=True)
    return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True) + FORCE_SUFFIX + suf

rollouts = []  # list of (question_idx, L11_emb_mean, correct_bool, predicted_letter)
t0 = time.time()

for qi, q in enumerate(tqdm(train_questions, desc='gen train rollouts')):
    p = format_prompt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1536: continue
    captured_l11['chunks'] = []
    try:
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                                 pad_token_id=tok.pad_token_id, use_cache=True)
        text = force_close(out[0], ids.shape[1])
        pred = extract_answer(text)
        correct = pred == q['gold_letter'] if pred else False
        # L11 residual: mean over response tokens
        chunks = captured_l11['chunks']
        if len(chunks) >= 2:
            emb = torch.stack(chunks[1:], dim=0).mean(dim=0)[0].float().numpy()  # [D_MODEL]
        else:
            continue
        rollouts.append({
            'question_idx': qi,
            'question': q['question'],
            'gold': q['gold_letter'],
            'pred': pred,
            'correct': correct,
            'emb': emb.tolist(),
        })
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue
    except Exception as e:
        print(f'  skip {qi}: {e}'); continue

    # Crash-safe: save every 10
    if (qi+1) % 10 == 0:
        with open(OUT / 'train_rollouts_partial.json', 'w') as f:
            json.dump(rollouts, f)
        nc = sum(1 for r in rollouts if r['correct'])
        print(f'  [{qi+1}/{len(train_questions)}] {len(rollouts)} kept, {nc} correct ({nc/max(1,len(rollouts))*100:.0f}%)')

elapsed = (time.time()-t0)/60
print(f'\n✅ {len(rollouts)} rollouts captured in {elapsed:.1f} min')
nc = sum(1 for r in rollouts if r['correct'])
print(f'Correctness rate: {nc}/{len(rollouts)} = {nc/max(1,len(rollouts))*100:.1f}%')

# Save final
with open(OUT / 'train_rollouts_final.json', 'w') as f:
    json.dump(rollouts, f)

## Cell 5 — 🚨 Upload corpus immediately (crash safety)

In [ ]:
create_repo(HF_OUT, repo_type='model', private=False, exist_ok=True)
api = HfApi()
api.upload_file(
    path_or_fileobj=str(OUT / 'train_rollouts_final.json'),
    path_in_repo='train_rollouts.json',
    repo_id=HF_OUT, repo_type='model',
    commit_message=f'27B rollouts corpus ({len(rollouts)} prompts, {nc} correct)'
)
print(f'✅ Corpus uploaded: https://huggingface.co/{HF_OUT}/blob/main/train_rollouts.json')

## Cell 6 — Train L11 probe on 27B residuals

In [ ]:
X = np.array([r['emb'] for r in rollouts])
y = np.array([int(r['correct']) for r in rollouts])
print(f'Probe training data: X {X.shape}, y {y.shape}, correct rate {y.mean():.3f}')

if len(np.unique(y)) < 2:
    print('⚠️ Only one class — need more diverse rollouts. Aborting probe train.')
else:
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    probe = LogisticRegression(max_iter=2000, C=0.5, class_weight='balanced')
    probe.fit(X_tr, y_tr)
    train_auc = roc_auc_score(y_tr, probe.predict_proba(X_tr)[:,1])
    val_auc = roc_auc_score(y_va, probe.predict_proba(X_va)[:,1])
    print(f'Probe trained on 27B L{PROBE_LAYER}:')
    print(f'  train AUROC: {train_auc:.3f}')
    print(f'  val AUROC:   {val_auc:.3f}')
    # For 35B-A3B we got val AUROC ~0.78; 27B should be similar or better
    with open(OUT / 'probe_l11.pkl', 'wb') as f:
        pickle.dump(probe, f)
    print('probe_l11.pkl saved')

## Cell 7 — 🚨 Upload probe immediately

In [ ]:
api.upload_file(
    path_or_fileobj=str(OUT / 'probe_l11.pkl'),
    path_in_repo='probe_l11.pkl',
    repo_id=HF_OUT, repo_type='model',
    commit_message=f'27B L11 LogReg probe (val AUROC {val_auc:.3f})'
)
print(f'✅ Probe uploaded')

## Cell 8 — PWMV + greedy generation wrappers

In [ ]:
def gen_greedy(ids):
    captured_l11['chunks'] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    return extract_answer(force_close(out[0], ids.shape[1]))

def gen_pwmv_all(ids, n_traces=N_TRACES, temp=TEMP):
    '''Returns (pwmv_answer, naive_vote, best_of_n, all_answers, all_scores)'''
    captured_l11['chunks'] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=temp,
                             num_return_sequences=n_traces, top_p=0.95,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    chunks = captured_l11['chunks']
    if len(chunks) >= 2:
        embs = torch.stack(chunks[1:], dim=0).mean(dim=0).numpy()
        scores = probe.predict_proba(embs)[:, 1].tolist()
    else:
        scores = [1.0] * n_traces
    answers = [extract_answer(force_close(out[i], ids.shape[1])) for i in range(n_traces)]
    # PWMV
    weighted = defaultdict(float)
    for a, s in zip(answers, scores):
        if a: weighted[a] += s
    pwmv = max(weighted, key=weighted.get) if weighted else None
    # Naive vote
    naive_c = Counter([a for a in answers if a])
    naive = naive_c.most_common(1)[0][0] if naive_c else None
    # Best-of-N
    best_i = int(np.argmax(scores))
    best = answers[best_i]
    return pwmv, naive, best, answers, scores

print('gen helpers ready')

## Cell 9 — Run 4-method eval on 20 held-out prompts (crash-safe)

In [ ]:
results = {'greedy': [], 'naive_vote': [], 'pwmv': [], 'best_of_n': []}
t0 = time.time()

for qi, q in enumerate(tqdm(eval_questions, desc='eval')):
    p = format_prompt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1536: continue
    gold = q['gold_letter']

    try:
        g = gen_greedy(ids)
        results['greedy'].append((g, g == gold))
    except Exception as e:
        print(f'  greedy err: {e}'); results['greedy'].append((None, False))

    try:
        pwmv, naive, bon, _, _ = gen_pwmv_all(ids)
        results['pwmv'].append((pwmv, pwmv == gold))
        results['naive_vote'].append((naive, naive == gold))
        results['best_of_n'].append((bon, bon == gold))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        for k in ['pwmv','naive_vote','best_of_n']:
            results[k].append((None, False))
    except Exception as e:
        print(f'  pwmv err: {e}')
        for k in ['pwmv','naive_vote','best_of_n']:
            results[k].append((None, False))

    # Crash-safe checkpoint every 3
    if (qi+1) % 3 == 0:
        print(f'  [{qi+1}/{len(eval_questions)}]', end=' ')
        for k in ['greedy','naive_vote','pwmv','best_of_n']:
            acc = np.mean([r[1] for r in results[k]])
            print(f'{k}={acc:.3f}', end=' ')
        print()
        partial = {k: [(r[0], bool(r[1])) for r in v] for k, v in results.items()}
        with open(OUT / 'eval_partial.json', 'w') as f:
            json.dump(partial, f, indent=2)

h_handle.remove()

print(f'\n=== FINAL (n={len(eval_questions)}, {(time.time()-t0)/60:.1f} min) ===')
for k in ['greedy','naive_vote','best_of_n','pwmv']:
    acc = np.mean([r[1] for r in results[k]])
    nones = sum(1 for r in results[k] if r[0] is None)
    print(f'{k:20s}: acc={acc:.3f}  None={nones}')

g = np.mean([r[1] for r in results['greedy']])
n = np.mean([r[1] for r in results['naive_vote']])
p = np.mean([r[1] for r in results['pwmv']])
b = np.mean([r[1] for r in results['best_of_n']])
print(f'\n=== DELTAS ===')
print(f'naive vote vs greedy: {(n-g)*100:+.1f}pp')
print(f'PWMV vs greedy:       {(p-g)*100:+.1f}pp')
print(f'PWMV vs naive:        {(p-n)*100:+.1f}pp')
print(f'best-of-N vs greedy:  {(b-g)*100:+.1f}pp')

## Cell 10 — Upload results + README

In [ ]:
summary = {
    'model': MODEL_ID,
    'n_train_rollouts': len(rollouts),
    'train_correct_rate': float(y.mean()),
    'probe_val_auroc': float(val_auc),
    'n_eval_prompts': len(eval_questions),
    'n_traces': N_TRACES,
    'temperature': TEMP,
    'max_new_tokens': MAX_NEW,
    'results': {
        'greedy': float(g), 'naive_vote': float(n),
        'pwmv': float(p), 'best_of_n': float(b),
    },
    'deltas_pp': {
        'naive_vs_greedy': float((n-g)*100),
        'pwmv_vs_greedy': float((p-g)*100),
        'pwmv_vs_naive': float((p-n)*100),
        'bon_vs_greedy': float((b-g)*100),
    },
    'official_qwen_baseline': {'SuperGPQA': 66.0},
    'claude_45_opus_baseline': {'SuperGPQA': 70.6},
}
with open(OUT / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

readme = f'''---
license: apache-2.0
base_model: Qwen/Qwen3.6-27B
tags:
- inference-wrapper
- probe-weighted-voting
- self-consistency
- reasoning
---

# qwen36-27b-deepconf-probe: PWMV on new Qwen3.6-27B dense flagship

Same probe-weighted voting wrapper as `qwen36-deepconf-probe` but retrained for the newly-released Qwen3.6-27B (2026-04-21).

## Results (SuperGPQA, n={len(eval_questions)} held-out)

| method | accuracy |
|---|---|
| greedy | {g:.3f} |
| naive vote (N={N_TRACES}) | {n:.3f} |
| best-of-N | {b:.3f} |
| **PWMV** | **{p:.3f}** |

Deltas: PWMV **{(p-g)*100:+.1f}pp** over greedy, {(p-n)*100:+.1f}pp over naive vote.

## Context

- Qwen team reported SuperGPQA 66.0 on 27B
- Claude 4.5 Opus SuperGPQA: 70.6
- Our wrapper target: close the gap

## Files

- `probe_l11.pkl` — L11 LogReg probe on 27B residuals (val AUROC {val_auc:.3f})
- `train_rollouts.json` — {len(rollouts)} labeled rollouts used to train probe
- `summary.json` — full benchmark breakdown

## Method (identical to 35B-A3B version)

1. Generate N={N_TRACES} traces with temp {TEMP}, max_new={MAX_NEW}
2. Force-close </think> if budget exceeded
3. Hook L{PROBE_LAYER} residual during generation
4. Mean-pool L{PROBE_LAYER} per trace → probe P(correct)
5. Weighted majority vote by probe confidence

## Related

- `caiovicentino1/qwen36-deepconf-probe` — 35B-A3B version (+6pp validated)
- `caiovicentino1/Qwen3.6-35B-A3B-LIMO-Probe` — LIMO LoRA stack (in progress)
'''
with open(OUT / 'README.md', 'w') as f: f.write(readme)

for fn in ['summary.json', 'README.md', 'eval_partial.json']:
    fp = OUT / fn
    if fp.exists():
        api.upload_file(path_or_fileobj=str(fp), path_in_repo=fn,
                        repo_id=HF_OUT, repo_type='model',
                        commit_message=f'upload {fn}')
        print(f'OK {fn}')

print(f'\n✅ Done: https://huggingface.co/{HF_OUT}')